In [19]:
from langgraph.graph import StateGraph ,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [20]:
load_dotenv()

True

In [21]:
model=ChatOpenAI()

In [22]:
class BlogState(TypedDict):

  title:str
  outline:str
  content:str
  

In [23]:
def create_outline(state: BlogState) -> BlogState:
    # Fetch title
    title = state["title"]

    # Call LLM
    prompt = f"Generate a detailed outline for a blog on the topic: {title}"

    outline = model.invoke(prompt).content

    # Update state
    state["outline"] = outline

    return state

In [24]:
def create_blog(state:BlogState)->BlogState:
  title=state['title']
  outline=state['outline']

  prompt=f"write a detailed blog on the title - {title} using the following outline \n {outline}"
  content=model.invoke(prompt).content
  
  state[content]=content

  return state



In [25]:
graph=StateGraph(BlogState)

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow=graph.compile()

In [26]:
intial_state={'title':'Rise of AI in India'}

In [27]:
workflow.invoke(intial_state)

{'title': 'Rise of AI in India',
 'outline': 'I. Introduction\n    A. Brief overview of Artificial Intelligence (AI)\n    B. Introduction to the rise of AI in India\n    C. Purpose of the blog\n\nII. Historical overview of AI in India\n    A. Early developments in AI in India\n    B. Adoption of AI in various sectors\n    C. Government initiatives to promote AI in India\n\nIII. Current state of AI in India\n    A. Major AI companies and startups in India\n    B. AI adoption in different industries such as healthcare, finance, and agriculture\n    C. Challenges faced by the AI industry in India\n\nIV. Role of government and policies in promoting AI in India\n    A. National AI strategy and initiatives\n    B. Regulation and policies concerning AI in India\n    C. Government support for AI research and development\n\nV. Impact of AI on the Indian economy and society\n    A. Advantages of AI in boosting productivity and efficiency\n    B. Potential job displacement due to AI automation\n 